# Analyzing Potential Relationships between Permit Clusters and Economic Indicators

## 1. Preliminaries

In [1]:
# Imports

# General

import numpy as np
import pandas as pd
import geopandas as gpd
import math
import json
import re
import string

# Plotting

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected' # For plotly graphs to render in this environment


# Scikit-Learn

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

In [2]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA"

In [3]:
# Load permits data

permits_df = pd.read_csv(f'{PATH}/PERMITS/permits.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

In [4]:
# Load economic data

econ_df = pd.read_csv(f'{PATH}/ECONOMIC/PROCESSED/full_economic_data.csv')

In [5]:
# Load geographic GEOJSON file

gdf = gpd.read_file("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/GEOGRAPHIC/PROCESSED/nbhds_with_zones.geojson")

# Assign correct coordinate system
gdf = gdf.set_crs('EPSG:3857', allow_override=True)

# Convert to desired coordinate system
gdf = gdf.to_crs(epsg=4326)

### 1.1 Helper functions

In [6]:
# Define function to examine dataframes

def examine_df(name,df,
               include_stats = True,
               include_sample = True):
    
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:\n")
    display(df.info())
    if include_stats == True:
        print(f'\n Basic statistical info about {name}:\n')
        display(df.describe())
    if include_sample == True:
        print(f"\n\nSample of records in the {name}:")
        display(df.head(5))

In [7]:
# Define function to generate correlation heatmap

def gen_corr_heatmap(df):

    '''
    Generate correlation heatmap for numeric columns of a dataframe
    '''
    
    num_df = df.select_dtypes(include = 'number') # Restrict to numeric columns
    corr_matrix = num_df.corr() # Compute correlation matrix
    
    fig = px.imshow(
        corr_matrix,
        text_auto=True, # Include text
        color_continuous_scale='RdBu', # Set color scale
        aspect='auto', # Set aspect ratio
        title='Correlation Heatmap of Numeric Columns',
        zmin=-1,   # force range
        zmax=1
    ) # Generate heatmap figure
    fig.update_layout(title={'x': 0.5})
    
    fig.update_layout(width=1100, height=1100)          # bigger figure
    fig.update_xaxes(tickangle=45, tickfont=dict(size=9))
    fig.update_yaxes(tickfont=dict(size=9))
        
    fig.show(renderer="notebook")